# Exercise 4: Transformers on Images + GLU-MLP Ablations (ViT × GLU Variants)

## In this exercise you will combine two influential ideas:

Vision Transformers (ViT) from “An Image is Worth 16×16 Words: Transformers for Image Recognition at Scale” (Dosovitskiy et al., 2020) https://arxiv.org/pdf/2010.11929:
ViT shows that you can treat an image like a sequence of tokens by splitting it into non-overlapping patches (e.g. 16×16 in the paper), embedding each patch into a vector, adding positional information, and then applying standard Transformer blocks for classification.

Gated MLPs (GLU variants) from “GLU Variants Improve Transformer” (Shazeer, 2020) https://arxiv.org/pdf/2002.05202:
Shazeer proposes replacing the standard Transformer feed-forward layer (FFN/MLP) with gated linear unit (GLU) variants such as GEGLU and SwiGLU, which often improves training dynamics and final performance under comparable compute/parameter budgets.

## What you will do

You will implement a tiny ViT-style classifier for MNIST, then run a controlled ablation where you replace the MLP inside each Transformer block:

Baseline FFN (GELU):
Linear(d_model → d_ff) → GELU → Linear(d_ff → d_model)

GLU-family MLPs (choose at least two and justify):

GEGLU, SwiGLU, other activation functions

Your goal is to evaluate whether these GLU variants change:

- convergence speed (loss vs steps),

- final test accuracy,

- and/or stability across runs.

## Key ViT concepts you will implement

- To convert MNIST images into Transformer tokens, you will:
  Patchify each 28×28 image into non-overlapping P×P patches.
  If P=4, then you get a 7×7 patch grid → 49 tokens per image.

- Embed patches with a linear layer: patch vectors → d_model.

- Add positional embeddings so the model knows where each patch came from.

- Apply n_layers Transformer encoder blocks.

- Pool token features (e.g., mean pooling) and project to 10 classes.

## Key GLU concept you will implement

GLU-style MLPs replace a standard FFN with a gating mechanism:
compute two projections a and b, apply a nonlinearity to a (variant-dependent), multiply elementwise: act(a) * b, project back to d_model.
To keep the comparison fair, use the 2/3 width rule from Shazeer.

What we provide vs what you implement

### We provide:

- MNIST loading + dataloaders

- a minimal training loop structure (AdamW)

- a suggested small model configuration that runs on CPU

### You implement:

- patch tokenization (patchify)

- patch embedding + positional embedding strategy

- a pre-LN Transformer encoder block using nn.MultiheadAttention

- at least two GLU MLP variants + one FFN baseline

- metric logging sufficient to support your conclusion

## Deliverables

Run at least 3 variants (baseline + the activation functions you choose for GLU) and report:

- final and best test accuracy

- number of trainable parameters

- a plot or printed summary of loss/accuracy over epochs

- a short discussion of your results

In [1]:
from __future__ import annotations

import math
from collections.abc import Callable
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
def patchify(x: torch.Tensor, patch_size: int) -> torch.Tensor:
    """Convert images to patch tokens.""" 
    # B: batch size 
    # C: number of color channel 
    # H: Height of image 
    # W: Width of image 
    B, C, H, W = x.shape 
    P = patch_size 

    assert H % P == 0 and W % P == 0, f"Image dimensions ({H}x{W}) must be divisible by patch_size ({P})" 

    h = H // P 
    w = W // P 
    # 1. Reshape into grid of patches: (B, C, h, p, w, p) 
    x = x.reshape(B, C, h, P, w, P) 
    # 2. Permute dimensions: (B, h, w, p, p, C) 
    x = torch.einsum("nchpwq->nhwpqc", x) 

    patches = x.reshape(B, h * w, P * P * C) 
    return patches 

In [3]:
# TODO: Add positional encoding as done in the ViT paper and patch projection 
class PatchEmbed(nn.Module): 
    def __init__(self, patch_dim: int, d_model: int): 
        super().__init__() 
        # patch_dim: previous token size 
        # d_model: token size 
        # Tranformer cần mọi token có cùng kích thước nên do đó cần linear projection để biến đổi về cùng một kích thước 
        self.proj = nn.Linear(patch_dim, d_model) 

    def forward(self, x_patches: torch.Tensor) -> torch.Tensor: 
        return self.proj(x_patches) 

class PositionalEmbedding(nn.Module): 
    def __init__(self, num_tokens: int, d_model: int): 
        super().__init__() 
        self.pos_embed = nn.Parameter( 
            torch.zeros(1, num_tokens, d_model) 
        ) 

    def forward(self, x: torch.Tensor) -> torch.Tensor: 
        return x + self.pos_embed

In [4]:
# TODO: Define the variants you want to compare against each other from the GLU paper. Justify your choice. 
class FeedForward(nn.Module): 
    """ 
    Standard Transformer FFN: 
    x -> Linear(d_model->d_ff) -> GELU -> Dropout -> Linear(d_ff->d_model) -> Dropout 
    """ 
    def __init__(self, d_model: int, d_ff: int, dropout: float): 
        super().__init__() 
        self.w1 = nn.Linear(d_model, d_ff) 
        self.act = nn.GELU() 
        self.dropout1 = nn.Dropout(dropout) 
        self.w2 = nn.Linear(d_ff, d_model) 
        self.dropout2 = nn.Dropout(dropout) 

    def forward(self, x: torch.Tensor) -> torch.Tensor: 
        x = self.w1(x) 
        x = self.act(x) 
        x = self.dropout1(x) 
        x = self.w2(x) 
        x = self.dropout2(x) 
        return x 

class GLUFeedForward(nn.Module): 
    """ 
    GLU-family FFN 
    GEGLU: 
    GELU(W_gate x) * (W_value x) 

    SwiGLU: 
    SiLU(W_gate x) * (W_value x) 

    Then project back to d_model. 
    """ 
    ACTIVATIONS: dict[str, Callable[[torch.Tensor], torch.Tensor]] = { 
        "swiglu": F.silu, # Swish_1 = SiLu 
        "geglu": F.gelu, # GELU 
        "reglu": F.relu, # ReLU 
        "glu": torch.sigmoid, # Standard Sigmoid GLU 
        "bilinear": lambda x: x # Identity (no activation) 
    } 
    def __init__(self, d_model: int, d_ff_gated: int, dropout: float, variant: str): 
        """ 
        Args: 
        d_model: Input/output dimension. 
        d_ff_gated: Hidden dimension (usually ~2/3 of standard d_ff). 
        dropout: Dropout probability. 
        variant: One of ['swiglu', 'geglu', 'reglu', 'glu', 'bilinear']. 
        bias: Whether to use bias (GLU paper sets bias=False). 
        """ 
        super().__init__() 
        variant = variant.lower() 
        if variant not in self.ACTIVATIONS: 
            raise ValueError(f"Unknown variant '{variant}'. Choose from {list(self.ACTIVATIONS.keys())}") 

        self.act = self.ACTIVATIONS[variant] 

        # W_gate (W in paper) and W_up (V in paper) 
        self.w_gate = nn.Linear(d_model, d_ff_gated) 
        self.w_up = nn.Linear(d_model, d_ff_gated) 

        # Intermediate dropout (applied to gated representation) 
        self.dropout1 = nn.Dropout(dropout) 

        # W_down 
        self.w_down = nn.Linear(d_ff_gated, d_model) 
        self.dropout2 = nn.Dropout(dropout) 

    def forward(self, x: torch.Tensor) -> torch.Tensor: 
        gate = self.act(self.w_gate(x)) 
        up = self.w_up(x) 
        hidden = gate * up 

        hidden = self.dropout1(hidden) 
        out = self.w_down(hidden) 
        out = self.dropout2(out) 
        return out 

In [5]:
class TransformerEncoderBlock(nn.Module): 
    """ 
    Pre-LN encoder block: 
    x = x + Dropout(SelfAttn(LN(x))) 
    x = x + Dropout(MLP(LN(x))) 
    """ 
    def __init__(self, d_model: int, n_heads: int, mlp: nn.Module, dropout: float): 
        super().__init__() 
        # Pre-attention normalization and self-attention 
        self.ln1 = nn.LayerNorm(d_model) 
        self.attn = nn.MultiheadAttention( 
            embed_dim = d_model, 
            num_heads = n_heads, 
            dropout = dropout, 
            batch_first = True # Expects input shape: (Batch, Seq_Len, Dim) 
        ) 
        self.dropout1 = nn.Dropout(dropout) 

        # Pre-MLP normalization and feed-forward network 
        self.ln2 = nn.LayerNorm(d_model) 
        self.mlp = mlp 
        self.dropout2 = nn.Dropout(dropout) 

    def forward(self, x: torch.Tensor) -> torch.Tensor: 
        """ 
        Args: 
        x: Input tensor of shape (B, seq_len, d_model) 
        attn_mask: Optional attention mask (typically None for standard ViT) 
        Returns: 
        Output tensor of shape (B, seq_len, d_model) 
        """ 
        # --- 1. Multi-Head Self-Attention Sub-layer (Pre-LN) --- 
        norm_x = self.ln1(x) 
        attn_out, _ = self.attn( 
            query = norm_x, 
            key = norm_x, 
            value = norm_x, 
            need_weights = False 
        ) 

        x = x + self.dropout1(attn_out) 

        # --- 2. Feed-Forward / MLP Sub-layer (Pre-LN) --- 
        norm_x = self.ln2(x) 
        mlp_out = self.mlp(norm_x) 
        x = x + self.dropout2(mlp_out) 
        return x

In [6]:
class TinyViT(nn.Module): 
    """ 
    Tiny ViT-style classifier for MNIST. 
    - patchify -> patch embed -> pos embed -> blocks -> mean pool -> head 
    """ 
    def __init__( 
        self, 
        patch_size: int, 
        d_model: int, 
        n_heads: int, 
        n_layers: int, 
        d_ff: int, 
        dropout: float, 
        mlp_kind: str, 
    ): 
        super().__init__() 
        assert 28 % patch_size == 0 
        grid = 28 // patch_size 
        self.num_tokens = grid * grid 
        self.patch_size = patch_size 
        patch_dim = patch_size * patch_size 

        # TODO: implement a strategy for embedding the patches 
        self.patch_embed = PatchEmbed(patch_dim=patch_dim, d_model=d_model) 
        self.pos_embed = PositionalEmbedding(num_tokens=self.num_tokens, d_model=d_model) 

        # TODO: implement a strategy to select the right mlp version for your experiment 
        if mlp_kind == "ffn": 
            def make_mlp(): 
                return FeedForward( 
                    d_model=d_model, 
                    d_ff=d_ff, 
                    dropout=dropout, 
                ) 
        else: 
            d_ff_gated = int(round(d_ff * 2 / 3)) 

            def make_mlp(): 
                return GLUFeedForward( 
                    d_model=d_model, 
                    d_ff_gated=d_ff_gated, 
                    dropout=dropout, 
                    variant=mlp_kind, 
                ) 

        # Transformer blocks 
        self.blocks = nn.ModuleList([ 
            TransformerEncoderBlock( 
                d_model=d_model, 
                n_heads=n_heads, 
                mlp=make_mlp(), 
                dropout=dropout, 
            ) 
            for _ in range(n_layers) 
        ]) 

        self.norm = nn.LayerNorm(d_model) 

        #MNIST has 10 classes 
        self.head = nn.Linear(d_model, 10) 

    def forward(self, x: torch.Tensor) -> torch.Tensor: 

        # [B, 1, 28, 28] 
        x = patchify(x, self.patch_size) 

        # [B, N, patch_dim] 
        x = self.patch_embed(x) 

        # [B, N, d_model] 
        x = self.pos_embed(x) 

        for block in self.blocks: 
            x = block(x) 

        x = self.norm(x) 
        # Mean pooling over tokens 
        x = x.mean(dim = 1) 

        logits = self.head(x) 
        return logits

In [7]:
@dataclass(frozen=True)
class TrainConfig:
    seed: int = 0
    batch_size: int = 128
    epochs: int = 3
    lr: float = 3e-4
    weight_decay: float = 0.01
    device: str = "cpu"  # set "cuda" if available

In [8]:
def train_one_run(
    mlp_kind: str,
    model: nn.Module,
    train_loader: DataLoader,
    test_loader: DataLoader,
    cfg: TrainConfig,
) -> dict:
    torch.manual_seed(cfg.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(cfg.seed)

    model.to(cfg.device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    train_losses: list[float] = []
    epoch_train_losses: list[float] = []
    test_accs: list[float] = []

    for epoch in range(cfg.epochs):

        # Train loop
        model.train()
        epoch_loss_sum = 0.0
        epoch_examples = 0
        for i, (xb, yb) in enumerate(train_loader):
            xb = xb.to(cfg.device)
            yb = yb.to(cfg.device)

            logits = model(xb)
            loss = F.cross_entropy(logits, yb)

            opt.zero_grad()
            loss.backward()
            opt.step()

            train_losses.append(loss.item())
            epoch_loss_sum += loss.item() * yb.size(0)
            epoch_examples += yb.size(0)

        epoch_train_losses.append(epoch_loss_sum / epoch_examples)

        # Evaluation loop NOTE: Should be no need to change this
        model.eval()
        correct = 0.0
        total = 0.0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(cfg.device)
                yb = yb.to(cfg.device)
                logits = model(xb)
                correct += (logits.argmax(dim=-1) == yb).float().sum().item()
                total += yb.numel()

        test_accs.append(correct / total)
        print(
            f"[{mlp_kind}] epoch {epoch+1}/{cfg.epochs} | "
            f"train loss: {epoch_train_losses[-1]:.4f} | "
            f"test acc: {test_accs[-1]:.4f}"
        )

    return {
        "mlp_kind": mlp_kind,
        "train_losses": train_losses,
        "epoch_train_losses": epoch_train_losses,
        "test_accs": test_accs,
        "final_test_acc": test_accs[-1],
        "best_test_acc": max(test_accs),
        "num_trainable_parameters": sum(
            parameter.numel() for parameter in model.parameters() if parameter.requires_grad
        ),
    }

In [9]:
cfg = TrainConfig(seed=0, batch_size=128, epochs=5, lr=3e-4, weight_decay=0.01, device="cpu")

tfm = transforms.Compose([transforms.ToTensor()])

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
test_ds = datasets.MNIST(root="./data", train=False, download=True, transform=tfm)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)

# Shared model configuration for a controlled ablation.
patch_size = 4
d_model = 64
n_heads = 4
n_layers = 2
d_ff = 256
dropout = 0.1

# Compare the GELU baseline with two GLU variants from the paper.
runs = ["ffn", "geglu", "swiglu"]
results = []

for kind in runs:
    # Re-seed before initialization so the comparison is reproducible.
    torch.manual_seed(cfg.seed)
    model = TinyViT(
        patch_size=patch_size,
        d_model=d_model,
        n_heads=n_heads,
        n_layers=n_layers,
        d_ff=d_ff,
        dropout=dropout,
        mlp_kind=kind,
    )
    num_parameters = sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )
    print(f"\nRun: {kind} | trainable parameters: {num_parameters:,}")
    out = train_one_run(kind, model, train_loader, test_loader, cfg)
    results.append(out)

print("\nAblation summary")
print("-" * 72)
print(f"{'variant':<12} {'parameters':>14} {'final acc':>14} {'best acc':>14}")
for result in results:
    print(
        f"{result['mlp_kind']:<12} "
        f"{result['num_trainable_parameters']:>14,} "
        f"{result['final_test_acc']:>13.2%} "
        f"{result['best_test_acc']:>13.2%}"
    )


Run: ffn | trainable parameters: 104,970
[ffn] epoch 1/5 | train loss: 1.1713 | test acc: 0.8087
[ffn] epoch 2/5 | train loss: 0.4119 | test acc: 0.8901
[ffn] epoch 3/5 | train loss: 0.2802 | test acc: 0.9259
[ffn] epoch 4/5 | train loss: 0.2213 | test acc: 0.9420
[ffn] epoch 5/5 | train loss: 0.1855 | test acc: 0.9353

Run: geglu | trainable parameters: 105,270
[geglu] epoch 1/5 | train loss: 1.0823 | test acc: 0.8548
[geglu] epoch 2/5 | train loss: 0.3541 | test acc: 0.9218
[geglu] epoch 3/5 | train loss: 0.2249 | test acc: 0.9369
[geglu] epoch 4/5 | train loss: 0.1794 | test acc: 0.9438
[geglu] epoch 5/5 | train loss: 0.1477 | test acc: 0.9574

Run: swiglu | trainable parameters: 105,270
[swiglu] epoch 1/5 | train loss: 1.0971 | test acc: 0.8525
[swiglu] epoch 2/5 | train loss: 0.3579 | test acc: 0.9178
[swiglu] epoch 3/5 | train loss: 0.2274 | test acc: 0.9412
[swiglu] epoch 4/5 | train loss: 0.1791 | test acc: 0.9495
[swiglu] epoch 5/5 | train loss: 0.1491 | test acc: 0.9610

Abl